# Modeling comparison for fraud detection

Notebook này so sánh 4 thuật toán:
- Logistic Regression
- Decision Tree
- Bayesian (GaussianNB + SVD)
- ANN (MLP + SVD)

Dữ liệu đọc từ `artifacts/processed_csv/` do `fraud_detection_btl.ipynb` xuất (ví dụ **20 cột** feature sau bước chọn `selected_features_model`).

- `X_train.csv` / `y_train.csv`: train **gốc** (imbalanced).
- `X_train_balanced.csv` / `y_train_balanced.csv`: train sau **2.8 SMOTE**.

Biến `USE_BALANCED_TRAIN`:
- `False`: chỉ huấn luyện trên tập **imbalanced**.
- `True`: huấn luyện **song song** trên **imbalanced** và **balanced** (cùng 4 thuật toán), bảng kết quả có cột `train_variant` để so sánh.

Metrics: `precision`, `recall`, `f1`, `roc_auc`, `pr_auc`, `accuracy` + confusion matrix + classification report.

Sau confusion matrix: **Visualization** (bar chart metric, đường ROC / Precision–Recall, heatmap confusion) → PNG trong `artifacts/modeling_results/figures/`.

Sau cell cuối: lưu thêm **mô hình đã fit** (`.pkl`, dùng `joblib`) trong `artifacts/modeling_results/trained_models/`.

In [ ]:
from pathlib import Path
import joblib
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import TruncatedSVD

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
TARGET = "is_fraud"


In [ ]:
# Load dữ liệu đã export từ notebook tiền xử lý
# True: huấn luyện song song imbalanced + balanced (SMOTE); False: chỉ imbalanced
USE_BALANCED_TRAIN = True

ARTIFACT_DIR = Path("../artifacts") if Path.cwd().name == "notebooks" else Path("artifacts")
processed_dir = ARTIFACT_DIR / "processed_csv"

required_files = [
    "X_train.csv", "y_train.csv",
    "X_valid.csv", "X_test.csv", "y_valid.csv", "y_test.csv",
]
if USE_BALANCED_TRAIN:
    required_files += ["X_train_balanced.csv", "y_train_balanced.csv"]

missing = [f for f in required_files if not (processed_dir / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Thiếu file trong {processed_dir.resolve()}: {missing}. "
        "Chạy cell export cuối fraud_detection_btl.ipynb; nếu USE_BALANCED_TRAIN=True cần đã chạy 2.8.2 trước export."
    )

X_train = pd.read_csv(processed_dir / "X_train.csv")
y_train = pd.read_csv(processed_dir / "y_train.csv")[TARGET]

X_train_balanced = y_train_balanced = None
if USE_BALANCED_TRAIN:
    X_train_balanced = pd.read_csv(processed_dir / "X_train_balanced.csv")
    y_train_balanced = pd.read_csv(processed_dir / "y_train_balanced.csv")[TARGET]
    assert list(X_train.columns) == list(X_train_balanced.columns), "Schema train imbalanced vs balanced không khớp"

X_valid = pd.read_csv(processed_dir / "X_valid.csv")
X_test = pd.read_csv(processed_dir / "X_test.csv")
y_valid = pd.read_csv(processed_dir / "y_valid.csv")[TARGET]
y_test = pd.read_csv(processed_dir / "y_test.csv")[TARGET]

print("Loaded from:", processed_dir.resolve())
print("Train imbalanced:", X_train.shape, "fraud rate", y_train.mean())
if USE_BALANCED_TRAIN:
    print("Train balanced:  ", X_train_balanced.shape, "fraud rate", y_train_balanced.mean())
print("Valid:", X_valid.shape, "| Test:", X_test.shape)
print("Fraud rate valid/test:", y_valid.mean(), y_test.mean())


In [ ]:
# Cột khớp với CSV từ fraud_detection_btl: object/category -> one-hot, còn lại -> numeric + scale
feature_columns = list(X_train.columns)
if TARGET in feature_columns:
    raise ValueError(f"X_train không được chứa cột target '{TARGET}'")

categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = [c for c in feature_columns if c not in categorical_features]

print(f"Số cột feature: {len(feature_columns)} (numeric={len(numeric_features)}, categorical={len(categorical_features)})")
print("Categorical:", categorical_features)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

transformers = [("num", numeric_pipeline, numeric_features)]
if categorical_features:
    transformers.append(("cat", categorical_pipeline, categorical_features))

base_preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")

# TruncatedSVD bắt buộc n_components <= số cột sau prep (ví dụ 20 feature số → không dùng được 60)
_probe_prep = clone(base_preprocessor)
_n_rows = min(20000, len(X_train))
_n_prep = _probe_prep.fit_transform(X_train.iloc[:_n_rows]).shape[1]
SVD_NB_COMPONENTS = max(1, min(60, _n_prep))
SVD_MLP_COMPONENTS = max(1, min(80, _n_prep))
print(f"Chiều sau preprocessor: {_n_prep} → TruncatedSVD: NB={SVD_NB_COMPONENTS}, MLP={SVD_MLP_COMPONENTS}")


In [ ]:
models = {
    "LogisticRegression": Pipeline([
        ("prep", base_preprocessor),
        ("clf", LogisticRegression(
            class_weight="balanced",
            max_iter=300,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
    "DecisionTree": Pipeline([
        ("prep", base_preprocessor),
        ("clf", DecisionTreeClassifier(
            max_depth=12,
            min_samples_leaf=20,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),
    "Bayesian_GaussianNB": Pipeline([
        ("prep", base_preprocessor),
        ("svd", TruncatedSVD(n_components=SVD_NB_COMPONENTS, random_state=RANDOM_STATE)),
        ("clf", GaussianNB()),
    ]),
    "ANN_MLP": Pipeline([
        ("prep", base_preprocessor),
        ("svd", TruncatedSVD(n_components=SVD_MLP_COMPONENTS, random_state=RANDOM_STATE)),
        ("clf", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            # learning_rate="adaptive",      # hoặc 'constant'
            # learning_rate_init=0.001,      # thử 0.01 
            early_stopping=True,
            max_iter=40,
            random_state=RANDOM_STATE,
        )),
    ]),
}

train_jobs = [("imbalanced", X_train, y_train)]
if USE_BALANCED_TRAIN:
    train_jobs.append(("balanced_smote", X_train_balanced, y_train_balanced))

artifacts = {}

for train_variant, Xt, yt in train_jobs:
    for name, model_template in models.items():
        print(f"\n=== [{train_variant}] Training {name} ===")
        model = clone(model_template)
        model.fit(Xt, yt)

        valid_prob = model.predict_proba(X_valid)[:, 1]
        test_prob = model.predict_proba(X_test)[:, 1]

        artifacts.setdefault(train_variant, {})[name] = {
            "fitted_model": model,
            "valid_prob": valid_prob,
            "test_prob": test_prob,
        }

print("Đã train và cache predict_proba vào artifacts. Đổi threshold chỉ cần chạy cell evaluate bên dưới.")

In [ ]:
# Đổi threshold tại đây, không cần train lại
threshold = 0.2

def evaluate_binary(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
    }

def evaluate_all(artifacts, y_valid, y_test, threshold):
    results = []
    for train_variant, models_dict in artifacts.items():
        for name, data in models_dict.items():
            valid_metrics = evaluate_binary(y_valid, data["valid_prob"], threshold)
            test_metrics = evaluate_binary(y_test, data["test_prob"], threshold)

            row = {
                "train_variant": train_variant,
                "model": name,
            }
            row.update({f"valid_{k}": v for k, v in valid_metrics.items()})
            row.update({f"test_{k}": v for k, v in test_metrics.items()})
            results.append(row)

    return pd.DataFrame(results)

results_df = evaluate_all(artifacts, y_valid, y_test, threshold)

sort_cols = ["valid_pr_auc", "valid_f1"]
if USE_BALANCED_TRAIN:
    sort_cols = ["train_variant"] + sort_cols
    asc = [True, False, False]
else:
    asc = [False, False]

results_df = results_df.sort_values(sort_cols, ascending=asc)
results_df

In [ ]:
# Bảng so sánh chính (validation + test); train_variant = imbalanced | balanced_smote
cols = [
    "train_variant", "model",
    "valid_precision", "valid_recall", "valid_f1", "valid_roc_auc", "valid_pr_auc",
    "test_precision", "test_recall", "test_f1", "test_roc_auc", "test_pr_auc",
]
display(results_df[cols].reset_index(drop=True))

if USE_BALANCED_TRAIN:
    for v in results_df["train_variant"].unique():
        sub = results_df[results_df["train_variant"] == v].sort_values(
            ["valid_pr_auc", "valid_f1"], ascending=False
        )
        best = sub.iloc[0]["model"]
        print(f"\nTốt nhất theo valid PR-AUC (train={v}): {best}")
else:
    sub = results_df.sort_values(["valid_pr_auc", "valid_f1"], ascending=False)
    print(f"\nMô hình tốt nhất theo valid PR-AUC: {sub.iloc[0]['model']}")


In [ ]:
# Confusion matrix + classification report (validation) theo từng train_variant + model
for train_variant in sorted(artifacts.keys()):
    for name in artifacts[train_variant]:
        valid_prob = artifacts[train_variant][name]["valid_prob"]
        y_pred = (valid_prob >= threshold).astype(int)

        cm = confusion_matrix(y_valid, y_pred)
        print(f"\n[{train_variant}] {name} - Validation confusion matrix")
        print(cm)
        print(f"\n[{train_variant}] {name} - Validation classification report")
        print(classification_report(y_valid, y_pred, digits=4, zero_division=0))


## Visualization

- **Bar chart**: so sánh precision, recall, F1, ROC-AUC, PR-AUC trên validation (và test) theo từng model / `train_variant`.
- **ROC / PR curves**: đường cong trên validation và test (xác suất lớp fraud; ROC/PR-AUC không phụ thuộc ngưỡng).
- **Confusion matrix (heatmap)**: nhất quán với ngưỡng `threshold` đã dùng khi train/evaluate.

Ảnh PNG được ghi vào `artifacts/modeling_results/figures/` với đuôi tên `threshold_<giá trị>` (cùng biến `threshold` như cell training). Chạy sau cell training (`results_df` / `artifacts`).

In [ ]:
# --- Visualization (chạy sau cell training) ---
fig_dir = ARTIFACT_DIR / "modeling_results" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
_thr_suffix = f"threshold_{float(threshold):g}"

id_cols = [c for c in ("train_variant", "model") if c in results_df.columns]
valid_metrics = ["valid_precision", "valid_recall", "valid_f1", "valid_roc_auc", "valid_pr_auc"]
test_metrics = ["test_precision", "test_recall", "test_f1", "test_roc_auc", "test_pr_auc"]

long_valid = results_df.melt(
    id_vars=id_cols,
    value_vars=valid_metrics,
    var_name="metric",
    value_name="value",
)
long_valid["metric"] = long_valid["metric"].str.replace("valid_", "", regex=False)

long_test = results_df.melt(
    id_vars=id_cols,
    value_vars=test_metrics,
    var_name="metric",
    value_name="value",
)
long_test["metric"] = long_test["metric"].str.replace("test_", "", regex=False)

# 1) Bar chart — validation
fig1, ax1 = plt.subplots(figsize=(11, 5))
if "train_variant" in long_valid.columns:
    long_valid["series"] = long_valid["train_variant"].astype(str) + " | " + long_valid["model"].astype(str)
    hue = "series"
else:
    long_valid["series"] = long_valid["model"]
    hue = "series"
sns.barplot(data=long_valid, x="metric", y="value", hue=hue, ax=ax1, palette="Set2")
ax1.set_title("Validation: precision / recall / F1 theo threshold; ROC-AUC & PR-AUC từ xác suất")
ax1.set_ylim(0, 1.05)
ax1.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
fig1.savefig(fig_dir / f"metrics_bar_valid_{_thr_suffix}.png", dpi=150, bbox_inches="tight")
plt.show()

# 1b) Bar chart — test
fig1b, ax1b = plt.subplots(figsize=(11, 5))
if "train_variant" in long_test.columns:
    long_test["series"] = long_test["train_variant"].astype(str) + " | " + long_test["model"].astype(str)
    hue_t = "series"
else:
    long_test["series"] = long_test["model"]
    hue_t = "series"
sns.barplot(data=long_test, x="metric", y="value", hue=hue_t, ax=ax1b, palette="Set2")
ax1b.set_title("Test metrics")
ax1b.set_ylim(0, 1.05)
ax1b.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
fig1b.savefig(fig_dir / f"metrics_bar_test_{_thr_suffix}.png", dpi=150, bbox_inches="tight")
plt.show()

# 2) ROC + PR curves (validation)
variants = sorted(artifacts.keys())
n_var = len(variants)
colors = plt.cm.tab10(np.linspace(0, 0.9, max(4, len(results_df))))

fig2, axes2 = plt.subplots(1, n_var, figsize=(5.5 * n_var, 4.5), squeeze=False)
for col, tv in enumerate(variants):
    ax = axes2[0, col]
    for i, (name, payload) in enumerate(artifacts[tv].items()):
        prob = payload["valid_prob"]
        fpr, tpr, _ = roc_curve(y_valid, prob)
        ax.plot(fpr, tpr, label=name, color=colors[i % len(colors)], linewidth=1.8)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.35, linewidth=1)
    ax.set_title(f"ROC — validation — {tv}")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.legend(fontsize=7, loc="lower right")
plt.tight_layout()
fig2.savefig(fig_dir / f"roc_valid_{_thr_suffix}.png", dpi=150, bbox_inches="tight")
plt.show()

fig3, axes3 = plt.subplots(1, n_var, figsize=(5.5 * n_var, 4.5), squeeze=False)
baseline = y_valid.mean()
for col, tv in enumerate(variants):
    ax = axes3[0, col]
    for i, (name, payload) in enumerate(artifacts[tv].items()):
        prob = payload["valid_prob"]
        prec, rec, _ = precision_recall_curve(y_valid, prob)
        ax.plot(rec, prec, label=name, color=colors[i % len(colors)], linewidth=1.8)
    ax.axhline(baseline, color="gray", linestyle="--", linewidth=1, label=f"baseline={baseline:.4f}")
    ax.set_title(f"Precision–Recall — validation — {tv}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.legend(fontsize=7, loc="upper right")
plt.tight_layout()
fig3.savefig(fig_dir / f"pr_valid_{_thr_suffix}.png", dpi=150, bbox_inches="tight")
plt.show()

# 3) ROC + PR — test
fig4, axes4 = plt.subplots(1, n_var, figsize=(5.5 * n_var, 4.5), squeeze=False)
for col, tv in enumerate(variants):
    ax = axes4[0, col]
    for i, (name, payload) in enumerate(artifacts[tv].items()):
        prob = payload["test_prob"]
        fpr, tpr, _ = roc_curve(y_test, prob)
        ax.plot(fpr, tpr, label=name, color=colors[i % len(colors)], linewidth=1.8)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.35, linewidth=1)
    ax.set_title(f"ROC — test — {tv}")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.legend(fontsize=7, loc="lower right")
plt.tight_layout()
fig4.savefig(fig_dir / f"roc_test_{_thr_suffix}.png", dpi=150, bbox_inches="tight")
plt.show()

fig5, axes5 = plt.subplots(1, n_var, figsize=(5.5 * n_var, 4.5), squeeze=False)
baseline_test = y_test.mean()
for col, tv in enumerate(variants):
    ax = axes5[0, col]
    for i, (name, payload) in enumerate(artifacts[tv].items()):
        prob = payload["test_prob"]
        prec, rec, _ = precision_recall_curve(y_test, prob)
        ax.plot(rec, prec, label=name, color=colors[i % len(colors)], linewidth=1.8)
    ax.axhline(baseline_test, color="gray", linestyle="--", linewidth=1, label=f"baseline={baseline_test:.4f}")
    ax.set_title(f"Precision–Recall — test — {tv}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.legend(fontsize=7, loc="upper right")
plt.tight_layout()
fig5.savefig(fig_dir / f"pr_test_{_thr_suffix}.png", dpi=150, bbox_inches="tight")
plt.show()

# 4) Confusion matrices (heatmap), cùng threshold với cell trên
thr = threshold
for tv in sorted(artifacts.keys()):
    items = list(artifacts[tv].items())
    ncols = len(items)
    fig_cm, axes_cm = plt.subplots(1, ncols, figsize=(3.4 * ncols, 3.2), squeeze=False)
    for j, (name, payload) in enumerate(items):
        ax = axes_cm[0, j]
        y_pred = (payload["valid_prob"] >= thr).astype(int)
        cm = confusion_matrix(y_valid, y_pred)
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            ax=ax,
            cbar=j == ncols - 1,
            xticklabels=["pred 0", "pred 1"],
            yticklabels=["true 0", "true 1"],
        )
        ax.set_title(name, fontsize=9)
    fig_cm.suptitle(f"Validation confusion matrix — {tv} (threshold={thr})", fontsize=11, y=1.02)
    plt.tight_layout()
    safe_tv = str(tv).replace(" ", "_")
    fig_cm.savefig(fig_dir / f"confusion_valid_{safe_tv}_{_thr_suffix}.png", dpi=150, bbox_inches="tight")
    plt.show()

print("Đã lưu figure tại:", fig_dir.resolve())

In [ ]:
# Lưu kết quả so sánh ra CSV + mô hình đã fit (.pkl)
out_dir = ARTIFACT_DIR / "modeling_results"
out_dir.mkdir(parents=True, exist_ok=True)
_thr_suffix = f"threshold_{float(threshold):g}"
comparison_path = out_dir / f"model_comparison_{_thr_suffix}.csv"
results_df.to_csv(comparison_path, index=False)
print("Saved:", comparison_path.resolve())

models_dir = out_dir / "trained_models"
models_dir.mkdir(parents=True, exist_ok=True)
for train_variant, by_name in artifacts.items():
    for name, payload in by_name.items():
        safe = f"{train_variant}__{name}".replace(" ", "_")
        path = models_dir / f"{safe}.pkl"
        joblib.dump(payload["fitted_model"], path)
        print("Model:", path.resolve())
print(f"Đã lưu {sum(len(v) for v in artifacts.values())} file .pkl tại: {models_dir.resolve()}")
